In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import pickle
import os

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print("Loaded:", df.shape)
df.head(3)

Loaded: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [2]:
df_clean = df.copy()

df_clean['TotalCharges'] = pd.to_numeric(
    df_clean['TotalCharges'], errors='coerce')

blanks = df_clean['TotalCharges'].isnull().sum()
print(f"Blank TotalCharges rows found: {blanks}")
df_clean.dropna(subset=['TotalCharges'], inplace=True)

df_clean.drop('customerID', axis=1, inplace=True)

df_clean['Churn'] = (df_clean['Churn'] == 'Yes').astype(int)

print(f"\nShape after cleaning: {df_clean.shape}")
print(f"Churn column: {df_clean['Churn'].value_counts().to_dict()}")
df_clean.dtypes

Blank TotalCharges rows found: 11

Shape after cleaning: (7032, 20)
Churn column: {0: 5163, 1: 1869}


gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

In [3]:
df_feat = df_clean.copy()

df_feat['charges_per_month'] = (
    df_feat['TotalCharges'] / (df_feat['tenure'] + 1)
).round(2)

df_feat['is_new_customer'] = (df_feat['tenure'] < 6).astype(int)

df_feat['is_long_term'] = (df_feat['tenure'] > 36).astype(int)

df_feat['has_multiple_services'] = (
    (df_feat['PhoneService'] == 'Yes').astype(int) +
    (df_feat['InternetService'] != 'No').astype(int) +
    (df_feat['StreamingTV'] == 'Yes').astype(int) +
    (df_feat['StreamingMovies'] == 'Yes').astype(int)
)

df_feat['high_value'] = (
    df_feat['MonthlyCharges'] > df_feat['MonthlyCharges'].median()
).astype(int)
print("New features added:")
print(df_feat[['charges_per_month', 'is_new_customer',
               'is_long_term', 'has_multiple_services',
               'high_value']].head())
print(f"\nDataset shape: {df_feat.shape}")

New features added:
   charges_per_month  is_new_customer  is_long_term  has_multiple_services  \
0              14.92                1             0                      1   
1              53.99                0             0                      2   
2              36.05                1             0                      2   
3              40.02                0             1                      1   
4              50.55                1             0                      2   

   high_value  
0           0  
1           0  
2           0  
3           0  
4           1  

Dataset shape: (7032, 25)


In [4]:
df_encoded = df_feat.copy()

cat_cols = df_encoded.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", cat_cols)

df_encoded = pd.get_dummies(df_encoded, columns=cat_cols, drop_first=True)

print(f"\nShape after encoding: {df_encoded.shape}")
print(f"Total features: {df_encoded.shape[1] - 1}")
df_encoded.head(2)

Categorical columns to encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Shape after encoding: (7032, 36)
Total features: 35


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,charges_per_month,is_new_customer,is_long_term,has_multiple_services,high_value,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,14.92,1,0,1,0,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,53.99,0,0,2,0,...,False,False,False,False,True,False,False,False,False,True


In [6]:
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Features: {X.shape[1]}")
print(f"Target distribution before SMOTE:")
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain size: {X_train.shape[0]} rows")
print(f"Test size:  {X_test.shape[0]} rows")
print(f"\nTrain churn distribution before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"\nTrain churn distribution AFTER SMOTE:")
print(pd.Series(y_train_res).value_counts())
print(f"New training size: {X_train_res.shape[0]} rows")
                                              

Features: 35
Target distribution before SMOTE:
Churn
0    5163
1    1869
Name: count, dtype: int64

Train size: 5625 rows
Test size:  1407 rows

Train churn distribution before SMOTE:
Churn
0    4130
1    1495
Name: count, dtype: int64



Train churn distribution AFTER SMOTE:
Churn
0    4130
1    4130
Name: count, dtype: int64
New training size: 8260 rows


In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

os.makedirs('../models', exist_ok=True)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv',   index=False)
pd.Series(y_train_res).to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

feature_cols = X.columns.tolist()
with open('../models/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)
    print("Saved:")
print("  models/scaler.pkl")
print("  models/feature_cols.pkl")
print("  data/X_train.csv  —", X_train_scaled.shape)
print("  data/X_test.csv   —", X_test_scaled.shape)
print("  data/y_train.csv  —", len(y_train_res), "rows")
print("  data/y_test.csv   —", len(y_test), "rows")
print("\nPreprocessing complete. Ready for modeling!")

Saved:
  models/scaler.pkl
  models/feature_cols.pkl
  data/X_train.csv  — (8260, 35)
  data/X_test.csv   — (1407, 35)
  data/y_train.csv  — 8260 rows
  data/y_test.csv   — 1407 rows

Preprocessing complete. Ready for modeling!
